# Sistema de Gestión de Cocina 

Este notebook contiene el diseño e implementación de un sistema de gestión de inventario para cocina. Se aplican los cuatro pilares fundamentales de la **Programación Orientada a Objetos (POO)** en Python: **Encapsulamiento, Herencia, Polimorfismo y Abstracción**, además de persistencia de datos mediante archivos `JSON`.

---

## Módulos Necesarios
Importación de las librerías estándar requeridas para el manejo del sistema operativo, serialización de datos JSON y formateo de fechas.

# Importaciones

In [36]:
import os
import json
from datetime import datetime

##  1. Modelo de Objetos (Jerarquía de Clases)

A continuación se definen las clases del dominio del problema:

* **`Producto` (Clase Base):** Aplica *encapsulamiento* utilizando atributos privados (`__nombre`, `__cantidad`, etc.) con sus respectivos decoradores `@property` y métodos manipuladores de estado (`sumar_cantidad`, `restar_cantidad`).
* **`ProductoAlacena` & `ProductoRefrigerado` (Subclases):** Aplican *herencia* al extender de `Producto` y *polimorfismo* al sobrescribir el método `obtener_detalle()`.
* **`Inventario` (Clase Gestora):** Colecciona y administra las instancias de `Producto`, permitiendo persistencia bidireccional con un archivo `JSON`.

In [37]:
class Producto:
    """Clase base que representa un producto genérico en la cocina.

    Attributes:
        nombre (str): Nombre del producto (formateado en capitalización).
        cantidad (float): Cantidad disponible del producto.
        unidad (str): Unidad de medida (kg, l, piezas, etc.).
        ubicacion (str): Lugar donde se almacena.
        fecha_ingreso (str): Fecha y hora de registro (YYYY-MM-DD HH:MM).
    """

    def __init__(self, nombre: str, cantidad: float, unidad: str, ubicacion: str, fecha_ingreso: str = None):
        self.__nombre = nombre.strip().capitalize()
        self.__cantidad = float(cantidad)
        self.__unidad = unidad.strip().lower()
        self.__ubicacion = ubicacion.strip().lower()
        self.__fecha_ingreso = fecha_ingreso or datetime.now().strftime("%Y-%m-%d %H:%M")

    # --- Getters y Setters ---
    @property
    def nombre(self) -> str:
        """str: Nombre del producto."""
        return self.__nombre

    @property
    def cantidad(self) -> float:
        """float: Cantidad disponible del producto."""
        return self.__cantidad

    @cantidad.setter
    def cantidad(self, nueva_cantidad: float):
        if nueva_cantidad >= 0:
            self.__cantidad = nueva_cantidad

    @property
    def unidad(self) -> str:
        """str: Unidad de medida."""
        return self.__unidad

    @property
    def ubicacion(self) -> str:
        """str: Ubicación del producto."""
        return self.__ubicacion

    @property
    def fecha_ingreso(self) -> str:
        """str: Fecha de ingreso del producto."""
        return self.__fecha_ingreso

    # --- Métodos de Operación ---
    def sumar_cantidad(self, extra: float):
        """Suma una cantidad positiva al stock actual."""
        if extra > 0:
            self.__cantidad += extra

    def restar_cantidad(self, consumo: float) -> bool:
        """Resta una cantidad del stock actual si hay suficiente existencia."""
        if 0 < consumo <= self.__cantidad:
            self.__cantidad -= consumo
            return True
        return False

    def obtener_detalle(self) -> str:
        """Retorna una cadena con el detalle básico del producto (Soporta Polimorfismo)."""
        return f"{self.__ubicacion.upper()} - {self.__nombre}: {self.__cantidad} {self.__unidad}"

    def to_dict(self) -> dict:
        """Serializa la instancia del objeto a un diccionario Python."""
        return {
            "tipo": self.__class__.__name__,
            "nombre": self.__nombre,
            "cantidad": self.__cantidad,
            "unidad": self.__unidad,
            "ubicacion": self.__ubicacion,
            "fecha_ingreso": self.__fecha_ingreso
        }


class ProductoAlacena(Producto):
    """Subclase de Producto para insumos almacenados a temperatura ambiente (Alacena)."""

    def __init__(self, nombre: str, cantidad: float, unidad: str, fecha_ingreso: str = None):
        super().__init__(nombre, cantidad, unidad, "alacena", fecha_ingreso)

    def obtener_detalle(self) -> str:
        """Sobrescribe el detalle agregando formato visual específico para Alacena."""
        return f"[ALACENA] 📦 {self.nombre} -> {self.cantidad} {self.unidad}"


class ProductoRefrigerado(Producto):
    """Subclase de Producto para insumos que requieren refrigeración."""

    def __init__(self, nombre: str, cantidad: float, unidad: str, fecha_ingreso: str = None):
        super().__init__(nombre, cantidad, unidad, "refrigerador", fecha_ingreso)

    def obtener_detalle(self) -> str:
        """Sobrescribe el detalle agregando formato visual específico para Refrigerador."""
        return f"[REFRI] ❄️ {self.nombre} -> {self.cantidad} {self.unidad}"


class Inventario:
    """Clase administradora encargada del almacenamiento y persistencia de productos."""

    def __init__(self, archivo: str = "inventario.json"):
        self.archivo = archivo
        self.productos = []
        self.cargar_inventario()

    def agregar(self, producto: Producto):
        """Agrega un nuevo producto o acumula cantidad si ya existe en el inventario."""
        existente = self.buscar(producto.nombre)
        if existente:
            existente.sumar_cantidad(producto.cantidad)
            print(f"El producto ya existía. Nueva cantidad acumulada: {existente.cantidad} {existente.unidad}")
        else:
            self.productos.append(producto)
            print(f"¡Producto agregado con éxito: {producto.nombre}!")

    def buscar(self, nombre: str) -> Producto | None:
        """Busca un producto por su nombre (insensible a mayúsculas/minúsculas)."""
        for p in self.productos:
            if p.nombre.lower() == nombre.lower():
                return p
        return None

    def listar_ordenado(self) -> list[Producto]:
        """Retorna la lista de productos ordenada por ubicación y nombre."""
        return sorted(self.productos, key=lambda p: (p.ubicacion, p.nombre))

    def guardar_inventario(self):
        """Guarda la lista de objetos serializados en formato JSON."""
        try:
            datos = [p.to_dict() for p in self.productos]
            with open(self.archivo, "w", encoding="utf-8") as f:
                json.dump(datos, f, indent=4, ensure_ascii=False)
            print(f"Inventario guardado exitosamente en {self.archivo}.")
        except Exception as e:
            print(f"Hubo un error al guardar el archivo: {e}")

    def cargar_inventario(self):
        """Carga e instancializa los productos desde el archivo JSON."""
        try:
            with open(self.archivo, "r", encoding="utf-8") as f:
                datos = json.load(f)
                self.productos = []
                for d in datos:
                    tipo = d.get("tipo", "Producto")
                    if tipo == "ProductoAlacena":
                        p = ProductoAlacena(d["nombre"], d["cantidad"], d["unidad"], d["fecha_ingreso"])
                    elif tipo == "ProductoRefrigerado":
                        p = ProductoRefrigerado(d["nombre"], d["cantidad"], d["unidad"], d["fecha_ingreso"])
                    else:
                        p = Producto(d["nombre"], d["cantidad"], d["unidad"], d["ubicacion"], d["fecha_ingreso"])
                    self.productos.append(p)
        except FileNotFoundError:
            print("No se encontró un archivo previo. Iniciando con inventario vacío.")
            self.productos = []
        except Exception:
            print("El archivo estaba dañado. Iniciando con inventario vacío.")
            self.productos = []

##  2. Funciones de Validación e Interfaz

Funciones independientes que facilitan la captura de datos por consola, validando entradas numéricas, cadenas no vacías y opciones de menú.

In [42]:
def limpiar_pantalla():
    """Limpia la consola del terminal dependiendo del sistema operativo."""
    os.system('cls' if os.name == 'nt' else 'clear')


def pedir_numero(mensaje: str) -> float:
    """Solicita un número flotante por consola asegurando que sea mayor a 0."""
    while True:
        try:
            numero = float(input(mensaje))
            if numero <= 0:
                print("El número tiene que ser mayor que cero.")
                continue
            return numero
        except ValueError:
            print("Eso no es un número válido. Intenta de nuevo.")


def pedir_texto(mensaje: str) -> str:
    """Solicita una cadena de texto asegurando que no quede vacía."""
    while True:
        texto = input(mensaje).strip()
        if texto == "":
            print("El campo no puede quedar vacío. Escribe algo.")
            continue
        return texto


def pedir_ubicacion() -> str:
    """Valida la entrada de ubicación restringida a 'alacena' o 'refrigerador'."""
    while True:
        lugar = input("¿Dónde está ubicado? (alacena/refrigerador): ").strip().lower()
        if lugar in ["alacena", "refrigerador"]:
            return lugar
        print("Opción no válida. Solo se permite 'alacena' o 'refrigerador'.")


def mostrar_menu() -> int:
    """Despliega el menú principal e interactivo por consola."""
    print("\n" + "=" * 40)
    print("      SISTEMA - GESTIONA TU COCINA")
    print("=" * 40)
    print("1. Ver inventario completo")
    print("2. Agregar nuevo producto")
    print("3. Consumir producto")
    print("4. Buscar producto específico")
    print("5. Ver productos por ubicación")
    print("6. Guardar y salir")
    print("=" * 40)

    while True:
        try:
            opcion = int(input("Elige una opción (1-6): "))
            if 1 <= opcion <= 6:
                return opcion
            print("Esa opción no existe. Debe ser un número del 1 al 6.")
        except ValueError:
            print("Por favor, ingresa un número entero válido.")

##  3. Demostración Interactiva (Sin Menú Consola)

En Jupyter Notebooks es muy práctico probar la lógica programática creando objetos directamente en celdas para validar el polimorfismo y la persistencia en JSON sin necesidad de bucles interactivos `while True`.

In [39]:
# Instanciamos un inventario de prueba
inv_demo = Inventario("inventario_demo.json")

# Creación de productos utilizando herencia
p1 = ProductoAlacena("Arroz", 5.0, "lbs")
p2 = ProductoRefrigerado("Leche", 2.0, "litros")

# Agregando productos
inv_demo.agregar(p1)
inv_demo.agregar(p2)

print("\n--- Demostración de Polimorfismo ---")
for p in inv_demo.listar_ordenado():
    # Invoca automáticamente el método de la subclase correspondiente
    print(p.obtener_detalle())

# Guardado en archivo
inv_demo.guardar_inventario()

El producto ya existía. Nueva cantidad acumulada: 35.0 lbs
El producto ya existía. Nueva cantidad acumulada: 14.0 litros

--- Demostración de Polimorfismo ---
[ALACENA] 📦 Arroz -> 35.0 lbs
[REFRI] ❄️ Leche -> 14.0 litros
Inventario guardado exitosamente en inventario_demo.json.


## 4. Ejecución Interactiva del Sistema Completo

Si deseas ejecutar la aplicación interactiva completa por terminal en la notebook, ejecuta la siguiente celda:

In [ ]:
def main():
    inventario = Inventario()

    while True:
        limpiar_pantalla()
        opcion = mostrar_menu()

        if opcion == 1:
            if not inventario.productos:
                print("\nNo hay nada en el inventario todavía.")
            else:
                print("\n" + "=" * 50)
                print("                  INVENTARIO GENERAL")
                print("=" * 50)
                for p in inventario.listar_ordenado():
                    print(p.obtener_detalle())
                print("=" * 50)
            input("\nPresiona Enter para continuar...")

        elif opcion == 2:
            print("\n--- AGREGAR PRODUCTO ---")
            nombre = pedir_texto("Nombre del producto: ")
            cantidad = pedir_numero("Cantidad: ")
            unidad = pedir_texto("Unidad de medida (kg, l, piezas, etc.): ")
            ubicacion = pedir_ubicacion()

            if ubicacion == "alacena":
                producto = ProductoAlacena(nombre, cantidad, unidad)
            else:
                producto = ProductoRefrigerado(nombre, cantidad, unidad)

            inventario.agregar(producto)
            input("\nPresiona Enter para continuar...")

        elif opcion == 3:
            print("\n--- CONSUMIR PRODUCTO ---")
            nombre = pedir_texto("Nombre del producto a consumir: ")
            producto = inventario.buscar(nombre)

            if not producto:
                print("No encontré ese producto en el inventario.")
            else:
                print(f"Producto: {producto.nombre}")
                print(f"Cantidad disponible: {producto.cantidad} {producto.unidad}")
                print(f"Ubicación: {producto.ubicacion}")

                cantidad_consumo = pedir_numero("¿Cuánto vas a consumir?: ")

                if cantidad_consumo > producto.cantidad:
                    print(f"No hay suficiente stock. Solo quedan {producto.cantidad} {producto.unidad}.")
                else:
                    producto.restar_cantidad(cantidad_consumo)
                    print(f"Operación exitosa. Queda un stock de: {producto.cantidad} {producto.unidad}")

                    if producto.cantidad == 0:
                        resp = input("El producto se agotó por completo. ¿Quieres borrarlo de la lista? (s/n): ").strip().lower()
                        if resp == 's':
                            inventario.productos.remove(producto)
                            print("El producto ha sido eliminado del inventario.")
            input("\nPresiona Enter para continuar...")

        elif opcion == 4:
            print("\n--- BUSCAR PRODUCTO ---")
            nombre = pedir_texto("Nombre del producto: ")
            producto = inventario.buscar(nombre)
            if producto:
                print("\n¡Lo encontré!")
                print(producto.obtener_detalle())
                print(f"Fecha de ingreso: {producto.fecha_ingreso}")
            else:
                print(f"No se encontró el producto '{nombre}'.")
            input("\nPresiona Enter para continuar...")

        elif opcion == 5:
            print("\n--- FILTRAR POR UBICACIÓN ---")
            ubicacion = pedir_ubicacion()
            encontrados = [p for p in inventario.productos if p.ubicacion == ubicacion]

            if not encontrados:
                print(f"No hay productos registrados en {ubicacion}.")
            else:
                print(f"\nProductos en {ubicacion.upper()}:")
                print("-" * 35)
                for p in encontrados:
                    print(f"- {p.nombre}: {p.cantidad} {p.unidad}")
                print("-" * 35)
                print(f"Total de registros: {len(encontrados)}")
            input("\nPresiona Enter para continuar...")

        elif opcion == 6:
            inventario.guardar_inventario()
            print("\n¡Nos vemos!")
            break

# Para ejecutar el menú interactivo, descomenta la siguiente línea:
#main()

No se encontró un archivo previo. Iniciando con inventario vacío.

      SISTEMA - GESTIONA TU COCINA
1. Ver inventario completo
2. Agregar nuevo producto
3. Consumir producto
4. Buscar producto específico
5. Ver productos por ubicación
6. Guardar y salir

--- AGREGAR PRODUCTO ---
¡Producto agregado con éxito: Huevo!

      SISTEMA - GESTIONA TU COCINA
1. Ver inventario completo
2. Agregar nuevo producto
3. Consumir producto
4. Buscar producto específico
5. Ver productos por ubicación
6. Guardar y salir
Inventario guardado exitosamente en inventario.json.

¡Nos vemos!
